In [23]:
import random
import collections
import heapq
from collections import deque
from typing import Tuple, List

def generate_branchy_maze(
    n: int,
    branchiness: float = 0.8,
    farthest_goal: bool = True
) -> Tuple[Tuple[int, int], Tuple[int, int], List[List[int]]]:
    """
    随机生成一张“分叉可调”的 perfect maze，并随机选取 start / goal。

    参数
    ----
    n            : 迷宫边长 (方形，单位格)
    branchiness  : 0‑1，越大分叉越多；≈0 -> DFS，≈1 -> Prim
    farthest_goal: True -> 选离 start 最远的格子当 goal；False -> 随机可达格

    返回
    ----
    (start, goal, maze)   其中 maze[i][j] == 0 表路, 1 表墙
    """
    # ---------- 初始化 ----------
    maze = [[1] * n for _ in range(n)]

    # start 随机挑一个偶数坐标 (保证格子间隔 2 时相邻仍在网格内)
    def rand_even(limit):                       # 0,2,4,… < limit
        max_even = limit - 1 if limit % 2 else limit - 2
        return random.randrange(0, max_even + 1, 2)

    start = (rand_even(n), rand_even(n))

    # ---------- Growing‑Tree 主循环 ----------
    def neighbors(x, y):
        # 返回: (邻居 x, 邻居 y, (wx, wy) 墙坐标增量)
        for dx, dy in [(0, 2), (2, 0), (0, -2), (-2, 0)]:
            nx, ny = x + dx, y + dy
            if 0 <= nx < n and 0 <= ny < n:
                yield nx, ny, dx // 2, dy // 2

    maze[start[0]][start[1]] = 0
    active = [start]

    while active:
        idx = -1 if random.random() > branchiness else random.randrange(len(active))
        x, y = active[idx]

        unvisited = [(nx, ny, wx, wy) for nx, ny, wx, wy in neighbors(x, y)
                     if maze[nx][ny] == 1]

        if unvisited:
            nx, ny, wx, wy = random.choice(unvisited)
            maze[x + wx][y + wy] = 0         # 打通墙
            maze[nx][ny] = 0
            active.append((nx, ny))
        else:
            active.pop(idx)                  # 死胡同：移除

    # ---------- 选取 goal ----------
    def bfs_farthest(src):
        """BFS 找到离 src 最远的可通行格；返回坐标"""
        vis = {src}
        q = deque([(src[0], src[1], 0)])
        far, far_dist = src, 0
        while q:
            x, y, d = q.popleft()
            if d > far_dist:
                far, far_dist = (x, y), d
            for dx, dy in [(0, 1), (1, 0), (0, -1), (-1, 0)]:
                nx, ny = x + dx, y + dy
                if 0 <= nx < n and 0 <= ny < n and maze[nx][ny] == 0 and (nx, ny) not in vis:
                    vis.add((nx, ny))
                    q.append((nx, ny, d + 1))
        return far

    if farthest_goal:
        goal = bfs_farthest(start)
    else:
        path_cells = [(i, j) for i in range(n) for j in range(n)
                      if maze[i][j] == 0 and (i, j) != start]
        goal = random.choice(path_cells)

    # 保证 start / goal 两格都是路
    maze[start[0]][start[1]] = 0
    maze[goal[0]][goal[1]]   = 0

    return start, goal, maze


# ────────────────────────── 2. 工具函数 ──────────────────────────
DIRS = [ (0, -1), (0, 1), (-1, 0), (1, 0) ]          # 左、右、上、下
def to1(p):                                           # 0‑based → 字符串 "(x, y)" 1‑based
    return f"({p[0] + 1}, {p[1] + 1})"

def observation_str(pos, maze, goal):
    n, (x, y) = len(maze), pos
    parts = []
    for dx, dy in DIRS:                               # 左→右→上→下
        nx, ny = x + dx, y + dy
        if 0 <= nx < n and 0 <= ny < n:
            if (nx, ny) == goal:
                state = "exit"
            else:
                state = "path" if maze[nx][ny] == 0 else "wall"
        else:
            state = "wall"
        parts.append(f"({nx + 1}, {ny + 1}): {state}")
    return "; ".join(parts)

# 已知网格内的 BFS 最短路（返回 deque，空表示不可达或就在原地）
def shortest_path(src, dst, walkable):
    if src == dst:
        return collections.deque()
    q = collections.deque([src])
    parent = {src: None}
    while q:
        cur = q.popleft()
        if cur == dst:
            break
        for dx, dy in DIRS:
            nxt = (cur[0] + dx, cur[1] + dy)
            if nxt in walkable and nxt not in parent:
                parent[nxt] = cur
                q.append(nxt)
    if dst not in parent:
        return collections.deque()
    path = collections.deque()
    cur = dst
    while cur != src:
        path.appendleft(cur)
        cur = parent[cur]
    return path


# ────────────────────────── 4. DEMO ──────────────────────────
if __name__ == "__main__":
    start, goal, maze = generate_branchy_maze(7)   # 或者用你自己的 maze
    print("Start:", start)
    print("End:", goal)
    for row in maze:
        print(row)


Start: (6, 6)
End: (0, 4)
[0, 0, 0, 0, 0, 1, 0]
[1, 1, 0, 1, 1, 1, 0]
[0, 1, 0, 0, 0, 1, 0]
[0, 1, 1, 1, 0, 1, 0]
[0, 0, 0, 0, 0, 1, 0]
[0, 1, 1, 1, 0, 1, 0]
[0, 1, 0, 0, 0, 0, 0]


In [24]:
def build_chat_history(maze, start, goal):
    """
    返回符合 OpenAI ChatCompletion 输入格式的 messages 列表：
    user ⇒ observation
    assistant ⇒ move
    """
    DIRS = [(0, -1), (0, 1), (-1, 0), (1, 0)]     # 左右上下

    def to1(p):                                    # (0‑based) → "(x, y)" (1‑based)
        return f"({p[0] + 1}, {p[1] + 1})"

    def obs_content(pos):
        n, (x, y) = len(maze), pos
        parts = []
        for dx, dy in DIRS:                        # 左→右→上→下
            nx, ny = x + dx, y + dy
            if 0 <= nx < n and 0 <= ny < n:
                state = (
                    "exit"  if (nx, ny) == goal else
                    "path"  if maze[nx][ny] == 0 else
                    "wall"
                )
            else:
                state = "wall"
            parts.append(f"({nx + 1}, {ny + 1}): {state}")
        return ", ".join(parts)                    # 用逗号分隔

    # ---------- 增量建图：与原 build_history_str 基本一致 ----------
    known, walls, frontier, task = {start}, set(), [], collections.deque()
    pos   = start
    gx, gy = goal
    messages = [{"role": "user", "content": obs_content(pos)}]

    def push_frontier(c):
        h = abs(c[0] - gx) + abs(c[1] - gy)
        heapq.heappush(frontier, (h, c))

    while True:
        # 1) 处理观测
        for dx, dy in DIRS:
            c = (pos[0] + dx, pos[1] + dy)
            if 0 <= c[0] < len(maze) and 0 <= c[1] < len(maze):
                if maze[c[0]][c[1]] == 0 or c == goal:
                    if c not in known:
                        known.add(c)
                        push_frontier(c)
                else:
                    walls.add(c)

        # 2) 结束
        if pos == goal:
            break

        # 3) 如无任务 ⇒ 规划
        if not task:
            if goal in known:
                task = shortest_path(pos, goal, known)
            while not task and frontier:
                _, tgt = heapq.heappop(frontier)
                task = shortest_path(pos, tgt, known)
            if not task:
                raise RuntimeError("No reachable target")

        # 4) 执行一步
        nxt = task.popleft()
        messages.append({"role": "assistant", "content": to1(nxt)})
        pos = nxt
        if pos != goal:                            # 到终点就不再发送观测
            messages.append({"role": "user", "content": obs_content(pos)})

    return messages


msgs = build_chat_history(maze, start, goal)

import json, pprint
pprint.pprint(msgs)                 # 直接看结构


[{'content': '(7, 6): path, (7, 8): wall, (6, 7): path, (8, 7): wall',
  'role': 'user'},
 {'content': '(6, 7)', 'role': 'assistant'},
 {'content': '(6, 6): wall, (6, 8): wall, (5, 7): path, (7, 7): path',
  'role': 'user'},
 {'content': '(5, 7)', 'role': 'assistant'},
 {'content': '(5, 6): wall, (5, 8): wall, (4, 7): path, (6, 7): path',
  'role': 'user'},
 {'content': '(4, 7)', 'role': 'assistant'},
 {'content': '(4, 6): wall, (4, 8): wall, (3, 7): path, (5, 7): path',
  'role': 'user'},
 {'content': '(3, 7)', 'role': 'assistant'},
 {'content': '(3, 6): wall, (3, 8): wall, (2, 7): path, (4, 7): path',
  'role': 'user'},
 {'content': '(2, 7)', 'role': 'assistant'},
 {'content': '(2, 6): wall, (2, 8): wall, (1, 7): path, (3, 7): path',
  'role': 'user'},
 {'content': '(1, 7)', 'role': 'assistant'},
 {'content': '(1, 6): wall, (1, 8): wall, (0, 7): wall, (2, 7): path',
  'role': 'user'},
 {'content': '(2, 7)', 'role': 'assistant'},
 {'content': '(2, 6): wall, (2, 8): wall, (1, 7): path,

In [25]:
def build_multi_trajectory_chat_history(maze, start, goal, K=3, N=10):
    """
    构造包含 K 个 trajectory 的 chat history，每个 trajectory 最多 N 步
    agent 可以利用前面所有 trajectory 收集到的信息
    """
    DIRS = [(0, -1), (0, 1), (-1, 0), (1, 0)]     # 左右上下

    def to1(p):
        return f"({p[0] + 1}, {p[1] + 1})"

    def obs_content(pos, traj_id):
        n, (x, y) = len(maze), pos
        parts = []
        for dx, dy in DIRS:
            nx, ny = x + dx, y + dy
            if 0 <= nx < n and 0 <= ny < n:
                state = (
                    "exit"  if (nx, ny) == goal else
                    "path"  if maze[nx][ny] == 0 else
                    "wall"
                )
            else:
                state = "wall"
            parts.append(f"({nx + 1}, {ny + 1}): {state}")
        obs_str = ", ".join(parts)
        return f"Trajectory {traj_id}: {obs_str}"

    n = len(maze)
    # Global knowledge that persists across all trajectories
    global_map = [[None for _ in range(n)] for _ in range(n)]
    global_map[start[0]][start[1]] = 0
    
    # Track all positions visited across all trajectories
    global_visited = set()
    
    # Track exploration frontiers (positions adjacent to unknown areas)
    exploration_frontiers = set()
    
    # Initialize messages with system prompt
    messages = [{"role": "system", "content": "You are an intelligent agent navigating a maze across multiple attempts. At each step, you receive an observation with trajectory ID and four adjacent cells (coordinates + 'path'/'wall'/'exit'). Learn from previous trajectories to navigate more efficiently. Choose exactly one adjacent 'path' or 'exit' cell to move into. Output your next move as coordinates (row, col) only."}]

    def update_global_map(pos):
        """Update global map with observations from current position"""
        for dx, dy in DIRS:
            nx, ny = pos[0] + dx, pos[1] + dy
            if 0 <= nx < n and 0 <= ny < n:
                if global_map[nx][ny] is None:
                    global_map[nx][ny] = maze[nx][ny]
                    # If it's a path, add it to exploration consideration
                    if maze[nx][ny] == 0:
                        exploration_frontiers.add((nx, ny))

    def get_known_walkable():
        """Get all currently known walkable positions"""
        return set(
            (i, j) for i in range(n) for j in range(n)
            if global_map[i][j] == 0 or (i, j) == goal
        )

    def get_exploration_targets(pos, known_walkable):
        """Get positions that are good for exploration (adjacent to unknown areas)"""
        targets = []
        for i in range(n):
            for j in range(n):
                if (i, j) in known_walkable:
                    # Check if this position has unknown neighbors
                    has_unknown_neighbor = False
                    for dx, dy in DIRS:
                        nx, ny = i + dx, j + dy
                        if 0 <= nx < n and 0 <= ny < n and global_map[nx][ny] is None:
                            has_unknown_neighbor = True
                            break
                    if has_unknown_neighbor and is_reachable(pos, (i, j), known_walkable):
                        targets.append((i, j))
        return targets

    def is_reachable(src, dst, walkable):
        if src == dst:
            return True
        q = collections.deque([src])
        visited = {src}
        while q:
            cur = q.popleft()
            if cur == dst:
                return True
            for dx, dy in DIRS:
                nxt = (cur[0] + dx, cur[1] + dy)
                if nxt in walkable and nxt not in visited:
                    visited.add(nxt)
                    q.append(nxt)
        return False

    def shortest_path(src, dst, walkable):
        if src == dst:
            return collections.deque()
        q = collections.deque([src])
        parent = {src: None}
        while q:
            cur = q.popleft()
            if cur == dst:
                break
            for dx, dy in DIRS:
                nxt = (cur[0] + dx, cur[1] + dy)
                if nxt in walkable and nxt not in parent:
                    parent[nxt] = cur
                    q.append(nxt)
        if dst not in parent:
            return collections.deque()
        path = collections.deque()
        cur = dst
        while cur != src:
            path.appendleft(cur)
            cur = parent[cur]
        return path

    def smart_next_move(pos, local_visited, known_walkable):
        """
        Smart strategy for choosing next move:
        1. Go directly to goal if known and reachable
        2. Explore unvisited known areas (prioritize closer to goal)
        3. Explore frontier areas (positions adjacent to unknown regions)
        4. Fall back to any reachable position
        """
        
        # Strategy 1: Direct path to goal if known and reachable
        if goal in known_walkable and goal not in local_visited:
            if is_reachable(pos, goal, known_walkable):
                path = shortest_path(pos, goal, known_walkable)
                if path:
                    return path.popleft()
        
        # Strategy 2: Visit unvisited known areas (prioritize by distance to goal)
        unvisited_known = [
            p for p in known_walkable 
            if p not in local_visited and p != pos and p not in global_visited
        ]
        if unvisited_known:
            # Sort by distance to goal
            target = min(unvisited_known, key=lambda p: abs(p[0] - goal[0]) + abs(p[1] - goal[1]))
            if is_reachable(pos, target, known_walkable):
                path = shortest_path(pos, target, known_walkable)
                if path:
                    return path.popleft()
        
        # Strategy 3: Explore areas that might reveal new information
        exploration_targets = get_exploration_targets(pos, known_walkable)
        if exploration_targets:
            # Prioritize targets closer to goal
            target = min(exploration_targets, 
                        key=lambda p: abs(p[0] - goal[0]) + abs(p[1] - goal[1]))
            if target not in local_visited:
                path = shortest_path(pos, target, known_walkable)
                if path:
                    return path.popleft()
        
        # Strategy 4: Visit any unvisited known position in this trajectory
        any_unvisited = [
            p for p in known_walkable 
            if p not in local_visited and p != pos
        ]
        if any_unvisited:
            target = min(any_unvisited, key=lambda p: abs(p[0] - goal[0]) + abs(p[1] - goal[1]))
            if is_reachable(pos, target, known_walkable):
                path = shortest_path(pos, target, known_walkable)
                if path:
                    return path.popleft()
                    
        return None

    for k in range(1, K + 1):
        pos = start
        trajectory_success = False
        local_visited = set()
        local_visited.add(pos)

        if messages[-1]["role"] == "user":
            messages[-1]["content"] = messages[-1]["content"] + "\n\n" + obs_content(pos, k)
        else:
            messages.append({"role": "user", "content": obs_content(pos, k)})
        update_global_map(pos)

        for step in range(N):
            update_global_map(pos)
            
            if pos == goal:
                trajectory_success = True
                break

            known_walkable = get_known_walkable()
            
            # Use smart strategy to choose next move
            next_pos = smart_next_move(pos, local_visited, known_walkable)
            
            if next_pos is None:
                # No valid moves available
                break
                
            # Execute the move
            messages.append({"role": "assistant", "content": to1(next_pos)})
            pos = next_pos
            local_visited.add(pos)
            global_visited.add(pos)  # Track globally visited positions
            
            # Continue with observation if not at goal and not last step
            if step < N - 1 and pos != goal:
                messages.append({"role": "user", "content": obs_content(pos, k)})

        # Add trajectory end message
        if trajectory_success:
            messages.append({"role": "user", "content": "Arrive the goal! Let's try again."})
        else:
            messages.append({"role": "user", "content": "You did not find the Exit. Let's try again."})

    return messages

# 使用示例
start, goal, maze = generate_branchy_maze(7)
chat_history = build_multi_trajectory_chat_history(maze, start, goal, K=5, N=20)

# 打印结果
for i, msg in enumerate(chat_history):
    print(f"{i}: {msg['role']}: {msg['content']}")

0: system: You are an intelligent agent navigating a maze across multiple attempts. At each step, you receive an observation with trajectory ID and four adjacent cells (coordinates + 'path'/'wall'/'exit'). Learn from previous trajectories to navigate more efficiently. Choose exactly one adjacent 'path' or 'exit' cell to move into. Output your next move as coordinates (row, col) only.
1: user: Trajectory 1: (1, 6): path, (1, 8): wall, (0, 7): wall, (2, 7): path
2: assistant: (1, 6)
3: user: Trajectory 1: (1, 5): path, (1, 7): path, (0, 6): wall, (2, 6): wall
4: assistant: (1, 5)
5: user: Trajectory 1: (1, 4): path, (1, 6): path, (0, 5): wall, (2, 5): path
6: assistant: (1, 4)
7: user: Trajectory 1: (1, 3): path, (1, 5): path, (0, 4): wall, (2, 4): wall
8: assistant: (1, 3)
9: user: Trajectory 1: (1, 2): path, (1, 4): path, (0, 3): wall, (2, 3): path
10: assistant: (1, 2)
11: user: Trajectory 1: (1, 1): path, (1, 3): path, (0, 2): wall, (2, 2): wall
12: assistant: (1, 1)
13: user: Trajec

In [4]:
msgs = []
for i in range(10000):
    start, goal, maze = generate_branchy_maze(7)
    msg = build_chat_history(maze, start, goal)
    msgs.append(msg)

In [10]:
print(msgs[0])

[{'role': 'user', 'content': '(1, 0): wall, (1, 2): path, (0, 1): wall, (2, 1): path'}, {'role': 'assistant', 'content': '(1, 2)'}, {'role': 'user', 'content': '(1, 1): path, (1, 3): path, (0, 2): wall, (2, 2): wall'}, {'role': 'assistant', 'content': '(1, 3)'}, {'role': 'user', 'content': '(1, 2): path, (1, 4): path, (0, 3): wall, (2, 3): path'}, {'role': 'assistant', 'content': '(1, 4)'}, {'role': 'user', 'content': '(1, 3): path, (1, 5): path, (0, 4): wall, (2, 4): wall'}, {'role': 'assistant', 'content': '(1, 5)'}, {'role': 'user', 'content': '(1, 4): path, (1, 6): path, (0, 5): wall, (2, 5): wall'}, {'role': 'assistant', 'content': '(1, 6)'}, {'role': 'user', 'content': '(1, 5): path, (1, 7): path, (0, 6): wall, (2, 6): wall'}, {'role': 'assistant', 'content': '(1, 7)'}, {'role': 'user', 'content': '(1, 6): path, (1, 8): wall, (0, 7): wall, (2, 7): wall'}, {'role': 'assistant', 'content': '(1, 6)'}, {'role': 'user', 'content': '(1, 5): path, (1, 7): path, (0, 6): wall, (2, 6): wal

In [23]:
import pandas as pd
from datasets import Dataset, Features, Sequence, Value

# 假设 msgs 是 list[list[dict]]
df = pd.DataFrame({"chat": msgs})


# 2️⃣ DataFrame ➜ Dataset
ds = Dataset.from_pandas(df)

In [4]:
import pandas as pd

# 假设 msgs = [ [ {"role":"user","content":"Hi"}, ... ],   # 第 0 条
#               [ {"role":"assistant","content":"Hello"}, ... ], ... ]

rows = [{'chat': m} for m in msgs]         # 每行是 {'chat': <list[dict]>}
df   = pd.DataFrame({'extra_info': rows})  # 只有一列 extra_info
df.to_parquet('/projectnb/rlhf/mingyuc/DisCO/datasets/maze/train100000.parquet',
              engine='pyarrow',           # 推荐，用 Arrow 写更快
              index=False)                # 不把行索引写进去


In [5]:
print(df)

                                              extra_info
0      {'chat': [{'role': 'user', 'content': '(5, 4):...
1      {'chat': [{'role': 'user', 'content': '(1, 4):...
2      {'chat': [{'role': 'user', 'content': '(7, 6):...
3      {'chat': [{'role': 'user', 'content': '(1, 4):...
4      {'chat': [{'role': 'user', 'content': '(5, 0):...
...                                                  ...
99995  {'chat': [{'role': 'user', 'content': '(5, 0):...
99996  {'chat': [{'role': 'user', 'content': '(5, 0):...
99997  {'chat': [{'role': 'user', 'content': '(3, 6):...
99998  {'chat': [{'role': 'user', 'content': '(1, 6):...
99999  {'chat': [{'role': 'user', 'content': '(1, 6):...

[100000 rows x 1 columns]


In [24]:
print(ds)

Dataset({
    features: ['chat'],
    num_rows: 10000
})


In [13]:
ds.push_to_hub(
    "MYC081/maze-sft"
)

Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.20it/s]


CommitInfo(commit_url='https://huggingface.co/datasets/MYC081/maze-sft/commit/9f8ec14685a0d56b833a681f22ae2f56e7640e2d', commit_message='Upload dataset', commit_description='', oid='9f8ec14685a0d56b833a681f22ae2f56e7640e2d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/MYC081/maze-sft', endpoint='https://huggingface.co', repo_type='dataset', repo_id='MYC081/maze-sft'), pr_revision=None, pr_num=None)

In [32]:
def generate_maze_rl_dataset(num_samples=1000, maze_size=7, data_source="maze_navigation", steps=20, episodes=5):
    """
    生成迷宫导航数据集，格式符合VERL训练要求
    """
    
    def observation_to_question(pos, maze, goal):
        """将当前位置的观测转换为问题描述"""
        DIRS = [(0, -1), (0, 1), (-1, 0), (1, 0)]  # 左右上下
        n, (x, y) = len(maze), pos
        parts = []
        for dx, dy in DIRS:
            nx, ny = x + dx, y + dy
            if 0 <= nx < n and 0 <= ny < n:
                state = (
                    "exit" if (nx, ny) == goal else
                    "path" if maze[nx][ny] == 0 else
                    "wall"
                )
            else:
                state = "wall"
            parts.append(f"({nx + 1}, {ny + 1}): {state}")
        
        return f"{', '.join(parts)}"
    
    def maze_to_solution(maze, start, goal):
        """将迷宫转换为解决方案字符串（包含迷宫布局和起点终点信息）"""
        solution_data = {
            "maze": maze,
            "start": start,
            "goal": goal,
            "size": len(maze)
        }
        return str(solution_data)
    
    def make_map_fn(split):
        def process_fn(example, idx):
            # 生成迷宫
            start, goal, maze = generate_branchy_maze(maze_size)
            
            # 生成起点观测作为问题
            question_raw = observation_to_question(start, maze, goal)
            question = "Trajectory 1: " + question_raw 
            
            
            data = {
                "data_source": data_source,
                "prompt": [
                    {
                        "role": "system",
                        "content": "You are an intelligent agent navigating a maze across multiple attempts. At each step, you receive an observation with trajectory ID and four adjacent cells (coordinates + 'path'/'wall'/'exit'). Learn from previous trajectories to navigate more efficiently. Choose exactly one adjacent 'path' or 'exit' cell to move into. Output your next move as coordinates (row, col) only.",
                    },
                    {
                        "role": "user",
                        "content": question,
                    },
                ],
                "ability": "navigation",  # 改为导航能力
                "reward_model": {"style": "rule", "ground_truth": goal},
                "extra_info": {
                    "split": split,
                    "index": idx,
                    "maze": maze,
                    "start": start,
                    "goal": goal,
                    "steps": steps,
                    "episodes": episodes,
                    "interaction_kwargs": {
                        "query": question,
                        "maze": maze,
                        "start": start,
                        "goal": goal,
                        "steps": steps,
                        "episodes": episodes,
                    },
                },
            }
            return data
        return process_fn
    
    # 生成数据集
    dataset = []
    process_fn = make_map_fn("train")
    
    avg_arrive = 0
    count = 0
    for i in range(num_samples):
        example = {}  # 空的example，因为我们直接在process_fn中生成迷宫
        data_item = process_fn(example, i)
        chat_history = build_multi_trajectory_chat_history(maze=data_item["extra_info"]["maze"],
                                           start=data_item["extra_info"]["start"],
                                           goal=data_item["extra_info"]["goal"],
                                           K=data_item["extra_info"]["episodes"],
                                           N=data_item["extra_info"]["steps"])
        arrive_numv = sum([1 for msg in chat_history if  "Arrive the goal! Let's try again." in msg['content']])

        if arrive_numv > 0 and arrive_numv < 6:
            dataset.append(data_item)
            avg_arrive += arrive_numv
            count += 1
        
        #if (i + 1) % 100 == 0:
            #print(f"Generated {i + 1}/{num_samples} samples...")
            #print(arrive_numv)

    print(f"Average arrive rate: {avg_arrive / count}")

    return dataset

# 生成数据集
maze_dataset = generate_maze_rl_dataset(num_samples=50000, maze_size=7, steps=20)
print(f"Generated {len(maze_dataset)} maze navigation samples")
print("\nSample data item:")
import pprint
pprint.pprint(maze_dataset[0])

Average arrive rate: 3.7324942791762012
Generated 48070 maze navigation samples

Sample data item:
{'ability': 'navigation',
 'data_source': 'maze_navigation',
 'extra_info': {'episodes': 5,
                'goal': (4, 0),
                'index': 0,
                'interaction_kwargs': {'episodes': 5,
                                       'goal': (4, 0),
                                       'maze': [[0, 1, 0, 0, 0, 0, 0],
                                                [0, 1, 1, 1, 1, 1, 0],
                                                [0, 0, 0, 0, 0, 0, 0],
                                                [1, 1, 1, 1, 0, 1, 0],
                                                [0, 1, 0, 0, 0, 1, 0],
                                                [0, 1, 1, 1, 1, 1, 0],
                                                [0, 0, 0, 0, 0, 0, 0]],
                                       'query': 'Trajectory 1: (1, 4): path, '
                                                '(1, 6): path, 

In [13]:
print(len(maze_dataset))

37711


In [22]:
# 保存数据集
import json
import pandas as pd
from datasets import Dataset


# 也可以转换为pandas DataFrame并保存为parquet
df = pd.DataFrame(maze_dataset)
df.to_parquet('/projectnb/rlhf/mingyuc/verl_github/verl/data/maze_w_interaction/test.parquet', 
              engine='pyarrow', 
              index=False)



In [ ]:
# 可选：生成大规模数据集
# 取消注释下面的代码来生成更大的数据集

# # 生成10000个样本的大数据集
# large_maze_dataset = generate_maze_dataset(num_samples=10000, maze_size=7)

# # 保存大数据集
# with open('/projectnb/rlhf/mingyuc/DisCO/datasets/maze/maze_dataset_10000.json', 'w', encoding='utf-8') as f:
#     json.dump(large_maze_dataset, f, ensure_ascii=False, indent=2)

# df_large = pd.DataFrame(large_maze_dataset)
# df_large.to_parquet('/projectnb/rlhf/mingyuc/DisCO/datasets/maze/maze_dataset_10000.parquet', 
#                     engine='pyarrow', 
#                     index=False)

# print(f"Large dataset with {len(large_maze_dataset)} samples saved!")

# 生成不同大小的迷宫数据集
def generate_mixed_size_dataset(num_samples=1000):
    """生成包含不同大小迷宫的混合数据集"""
    mixed_dataset = []
    sizes = [5, 7, 9, 11]  # 不同的迷宫大小
    samples_per_size = num_samples // len(sizes)
    
    for size in sizes:
        print(f"Generating {samples_per_size} samples with maze size {size}x{size}")
        size_dataset = generate_maze_dataset(num_samples=samples_per_size, maze_size=size)
        mixed_dataset.extend(size_dataset)
    
    # 重新索引
    for i, item in enumerate(mixed_dataset):
        item['extra_info']['index'] = i
    
    return mixed_dataset

# 取消注释下面的代码来生成混合大小的数据集
# mixed_dataset = generate_mixed_size_dataset(num_samples=2000)
# with open('/projectnb/rlhf/mingyuc/DisCO/datasets/maze/maze_mixed_dataset.json', 'w', encoding='utf-8') as f:
#     json.dump(mixed_dataset, f, ensure_ascii=False, indent=2)
# print(f"Mixed size dataset with {len(mixed_dataset)} samples saved!")

In [15]:
import os
import json
import pandas as pd
from datasets import Dataset

# 创建保存目录
local_dir = "./data/maze"
os.makedirs(local_dir, exist_ok=True)

print("Generating maze datasets...")

# 生成训练集 (10000 samples)
print("Generating training set...")
train_dataset_raw = generate_maze_rl_dataset(num_samples=50000, maze_size=7, data_source="maze_navigation", steps=15)

# 生成测试集 (500 samples)
print("Generating test set...")
test_dataset_raw = generate_maze_rl_dataset(num_samples=500, maze_size=7, data_source="maze_navigation", steps=15)

# 更新split标记
for item in train_dataset_raw:
    item['extra_info']['split'] = 'train'

for item in test_dataset_raw:
    item['extra_info']['split'] = 'test'

print(f"Generated {len(train_dataset_raw)} training samples and {len(test_dataset_raw)} test samples")

# 转换为HuggingFace Dataset格式
train_df = pd.DataFrame(train_dataset_raw)
test_df = pd.DataFrame(test_dataset_raw)

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# 保存为parquet格式（按照GSM8K格式）
train_dataset.to_parquet(os.path.join(local_dir, "train50000.parquet"))
test_dataset.to_parquet(os.path.join(local_dir, "test.parquet"))

print(f"Datasets saved successfully!")
print(f"Training set: {os.path.join(local_dir, 'train50000.parquet')} ({len(train_dataset)} samples)")
print(f"Test set: {os.path.join(local_dir, 'test.parquet')} ({len(test_dataset)} samples)")

# 验证保存的数据
print(f"\nTraining dataset info:")
print(f"  Features: {list(train_dataset.features.keys())}")
print(f"  Sample count: {len(train_dataset)}")
print(f"\nTest dataset info:")
print(f"  Features: {list(test_dataset.features.keys())}")
print(f"  Sample count: {len(test_dataset)}")

# 显示一个训练样本作为验证
print(f"\nSample training item:")
print(f"Data source: {train_dataset[0]['data_source']}")
print(f"Ability: {train_dataset[0]['ability']}")
print(f"Prompt messages: {len(train_dataset[0]['prompt'])}")
print(f"User message: {train_dataset[0]['prompt'][1]['content'][:100]}...")
print(f"Split: {train_dataset[0]['extra_info']['split']}")
print(f"Index: {train_dataset[0]['extra_info']['index']}")
print(f"Maze size: {len(train_dataset[0]['extra_info']['maze'])}x{len(train_dataset[0]['extra_info']['maze'][0])}")
print(f"Start: {train_dataset[0]['extra_info']['start']}")
print(f"Goal: {train_dataset[0]['extra_info']['goal']}")

Generating maze datasets...
Generating training set...
Generated 100/50000 samples...
Generated 200/50000 samples...
Generated 300/50000 samples...
Generated 400/50000 samples...
Generated 500/50000 samples...
Generated 600/50000 samples...
Generated 700/50000 samples...
Generated 800/50000 samples...
Generated 900/50000 samples...
Generated 1000/50000 samples...
Generated 1100/50000 samples...
Generated 1200/50000 samples...
Generated 1300/50000 samples...
Generated 1400/50000 samples...
Generated 1500/50000 samples...
Generated 1600/50000 samples...
Generated 1700/50000 samples...
Generated 1800/50000 samples...
Generated 1900/50000 samples...
Generated 2000/50000 samples...
Generated 2100/50000 samples...
Generated 2200/50000 samples...
Generated 2300/50000 samples...
Generated 2400/50000 samples...
Generated 2500/50000 samples...
Generated 2600/50000 samples...
Generated 2700/50000 samples...
Generated 2800/50000 samples...
Generated 2900/50000 samples...
Generated 3000/50000 sampl

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 562.77ba/s]

Datasets saved successfully!
Training set: ./data/maze/train50000.parquet (37655 samples)
Test set: ./data/maze/test.parquet (369 samples)

Training dataset info:
  Features: ['data_source', 'prompt', 'ability', 'reward_model', 'extra_info']
  Sample count: 37655

Test dataset info:
  Features: ['data_source', 'prompt', 'ability', 'reward_model', 'extra_info']
  Sample count: 369

Sample training item:
Data source: maze_navigation
Ability: navigation
Prompt messages: 2
User message: Trajectory 1: (1, 6): path, (1, 8): wall, (0, 7): wall, (2, 7): path...
Split: train
Index: 0
Maze size: 7x7
Start: [0, 6]
Goal: [6, 0]


In [17]:
print(train_dataset[0]['extra_info']['interaction_kwargs'])

{'goal': [6, 4], 'ground_truth': "{'maze': [[0, 0, 0, 0, 0, 0, 0], [0, 1, 1, 1, 0, 1, 0], [0, 1, 0, 0, 0, 1, 0], [0, 1, 0, 1, 1, 1, 0], [0, 1, 0, 1, 0, 0, 0], [0, 1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 1, 0]], 'start': (2, 2), 'goal': (6, 4), 'size': 7}", 'maze': [[0, 0, 0, 0, 0, 0, 0], [0, 1, 1, 1, 0, 1, 0], [0, 1, 0, 0, 0, 1, 0], [0, 1, 0, 1, 1, 1, 0], [0, 1, 0, 1, 0, 0, 0], [0, 1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 1, 0]], 'query': 'Trajectory 1: (3, 2): wall, (3, 4): path, (2, 3): wall, (4, 3): path ', 'start': [2, 2]}
